# 🏆 Pokémon Battle Outcome Prediction — Competitive Type-Trajectory & Metagame Pipeline
**Course:** Fundamentals / Foundations of Data Science (Sapienza University of Rome, I Semester 2026–2027)  
**Format:** Pokémon Generation 2 OverUsed (GSC OU) — Binary Classification (`player_won` $\in \{0, 1\}$)

---

## 1. Executive Summary & Core Hypothesis

In this challenge, we must predict the final winner of a competitive Generation 2 OU Pokémon battle (`player_won`: `1` if Player 1 wins, `0` if Player 1 loses) while observing **only the first 30 active turns** of matches that last between **30 and 80 turns**. Furthermore, while we know all 6 Pokémon on Player 1's roster (`p1_team_details`), we only initially see Player 2's starting Pokémon (`p2_lead_details`) plus whichever Pokémon Player 2 reveals during the first 30 turns (`battle_timeline`).

Rather than relying solely on simple turn-30 HP snapshots—which fail when a player is setting up a late-game comeback or holding an unrevealed counter on the bench—this notebook implements a **Two-Stage Competitive Type-Trajectory & Metagame Architecture** built around four domain insights:

1. **Standardized Competitive Metagame (Internal Knowledge Mining):**  
   Because all battles take place in the competitive **Gen 2 OU** tier, team compositions and movesets are highly standardized (e.g., *Snorlax* appears in ~82% of teams, *Cloyster* in ~54%, *Zapdos* in ~52%). By mining `p1_team_details` and `battle_timeline` strictly from the training data, we construct an internal Pokédex, Pokémon usage priors, standard moveset profiles, and a **Team Type Co-occurrence Matrix** to infer the probability distribution of Player 2's unrevealed bench types.
2. **Move Effectiveness, Resistance & Setup Dynamics Drive Trainer Switching:**  
   In competitive play, trainers do not switch randomly. A trainer **stays in** if their active Pokémon resists the opponent's attack ($\le 0.5\times$), has sufficient HP to tank a hit, or acts as a **Setup / Hazard / Status** enabler (e.g., *Curse*, *Spikes*, *Rest + Sleep Talk*, *Roar/Whirlwind*). Conversely, facing a super-effective threat ($\ge 2.0\times$) forces a switch into a resistant bench type.
3. **Predicting the Next Type (Turn-31 Horizon Forecasting):**  
   Since the observation window cuts off sharply at Turn 30, knowing *who is forced to switch on Turn 31* and *which Pokémon types will take control of the field next* provides a critical lookahead signal beyond the 30-turn horizon.
4. **Whole-Team Remaining Type Dominance:**  
   Matches lasting 30–80 turns are decided by whether Player 1's surviving Pokémon types can break through Player 2's surviving + predicted hidden types using the exact Generation 2 Type Chart multipliers (`0.0x`, `0.25x`, `0.5x`, `1.0x`, `2.0x`, `4.0x`).

> **Rule Compliance Statement:**  
> This notebook strictly adheres to all competition rules:
> * **100% Classical Machine Learning:** Uses only tree-based gradient boosting and linear stacking (`LightGBM`, `XGBoost`, `CatBoost`, `LogisticRegression`). No neural networks or deep learning models are used.
> * **Zero External Data Leakage:** All usage rates, moveset profiles, and type co-occurrences are learned exclusively from the provided competition files (`train.jsonl` and `type_chart_gen2.csv`).

---

## 2. Notebook Architecture & Code Structure

The pipeline is organized into **4 modular sequential stages**:

```text
├── STEP 1: Environment Setup, Data Loading & Internal Metagame Knowledge Base
│   ├── Step 1A: Imports, constants (17 Gen-2 types), and directory path configuration
│   ├── Step 1B: Unified Data Loader (parses train.jsonl / test.jsonl / unpacked data)
│   └── Step 1C: CompetitiveMetaKnowledgeBase class (Pokedex, Usage Priors, Type Co-occurrence, Setup Profiler)
│
├── STEP 2: Turn-by-Turn Matchup Engine & Stage-1 Next-Type / Switch Predictor
│   ├── Step 2A: Turn-level feature extraction (move multipliers, expected threat, resistances, bench counters)
│   └── Step 2B: Stage-1 Auxiliary ML Models (LightGBM Switch Classifiers + Multi-Output Next-Type Classifiers)
│
├── STEP 3: Turn-31 Horizon Forecasting & Battle-Level Feature Aggregation
│   ├── Step 3A: Turn-31 Next-Type probability projection & expected type matchup matrix
│   └── Step 3B: Full 30-turn battle aggregation (HP pools, status, Spikes, momentum, remaining team type dominance)
│
└── STEP 4: Stage-2 Ensemble Training, Stratified K-Fold Validation & Submission
    ├── Step 4A: Test set (`test.jsonl`) transformation through the Step 2 & Step 3 pipelines
    ├── Step 4B: Stratified K-Fold Cross-Validation (`LightGBM` + `XGBoost` + `CatBoost`)
    └── Step 4C: Out-of-Fold (OOF) diagnostics, threshold calibration, feature importance & `submission.csv` export

In [ ]:
# ==============================================================================
# STEP 1A: IMPORTS, CONFIGURATION & FILE PATHS
# ==============================================================================
import os
import json
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# If using Google Drive on Colab, uncomment the next two lines:
# from google.colab import drive
# drive.mount('/content/drive')

# Set your base directory path (keep the same path structure you use on Colab/Kaggle)
DATA_DIR = "/content/drive/MyDrive/1 SEMESTRE DATA SCIENCE/FUNDAMENTALS OF DATA SCIENCE/FDS PROJECT"  # Change to your Drive/Kaggle folder if needed, e.g., "/content/drive/MyDrive/FDS_Pokemon/"

TYPE_CHART_PATH = os.path.join(DATA_DIR, "type_chart_gen2.csv")
EXCEL_UNPACKED_PATH = os.path.join(DATA_DIR, "pokemon_json_spacchettato.xlsx")
TRAIN_JSONL_PATH = os.path.join(DATA_DIR, "train.jsonl")
TEST_JSONL_PATH = os.path.join(DATA_DIR, "test.jsonl")

ALL_TYPES = [
    'normal', 'fire', 'water', 'electric', 'grass', 'ice', 'fighting',
    'poison', 'ground', 'flying', 'psychic', 'bug', 'rock', 'ghost',
    'dragon', 'dark', 'steel'
]

In [ ]:
# ==============================================================================
# STEP 1B: DATA LOADER (Supports both unpacked Excel and raw JSONL)
# ==============================================================================
def load_competition_data():
    """
    Loads p1_team_details, battle_timeline, and p2_lead_details from either
    train.jsonl (if available) or pokemon_json_spacchettato.xlsx.
    """
    if os.path.exists(EXCEL_UNPACKED_PATH) and not os.path.exists(TRAIN_JSONL_PATH):
        print(f"Loading unpacked data from {EXCEL_UNPACKED_PATH}...")
        xls = pd.ExcelFile(EXCEL_UNPACKED_PATH)
        p1_team_df = pd.read_excel(xls, 'p1_team_details')
        timeline_df = pd.read_excel(xls, 'battle_timeline')
        p2_lead_df = pd.read_excel(xls, 'p2_lead_details')
        return p1_team_df, timeline_df, p2_lead_df

    elif os.path.exists(TRAIN_JSONL_PATH):
        print(f"Loading raw JSONL data from {TRAIN_JSONL_PATH}...")
        p1_rows, timeline_rows, p2_rows = [], [], []
        with open(TRAIN_JSONL_PATH, 'r') as f:
            for line in f:
                b = json.loads(line)
                b_id = b['battle_id']

                # P2 Lead & Target
                p2 = b['p2_lead_details'].copy()
                p2['battle_id'] = b_id
                p2['player_won'] = int(b['player_won'])
                p2_rows.append(p2)

                # P1 Team (6 Pokemon)
                for pkmn in b['p1_team_details']:
                    p1 = pkmn.copy()
                    p1['battle_id'] = b_id
                    p1_rows.append(p1)

                # Timeline (Up to 30 turns)
                for turn_data in b['battle_timeline']:
                    row = {'battle_id': b_id, 'turn': turn_data['turn']}
                    for side in ['p1', 'p2']:
                        state = turn_data.get(f'{side}_pokemon_state', {}) or {}
                        row[f'{side}_pokemon_state.name'] = state.get('name')
                        row[f'{side}_pokemon_state.hp_pct'] = state.get('hp_pct')
                        row[f'{side}_pokemon_state.status'] = state.get('status')
                        row[f'{side}_pokemon_state.effects'] = str(state.get('effects', 'noeffect'))
                        boosts = state.get('boosts', {}) or {}
                        for stat in ['atk', 'def', 'spa', 'spd', 'spe']:
                            row[f'{side}_pokemon_state.boosts.{stat}'] = boosts.get(stat, 0)

                        move = turn_data.get(f'{side}_move_details', {}) or {}
                        row[f'{side}_move_details.name'] = move.get('name')
                        row[f'{side}_move_details.type'] = move.get('type')
                        row[f'{side}_move_details.category'] = move.get('category')
                        row[f'{side}_move_details.base_power'] = move.get('base_power')
                        row[f'{side}_move_details.accuracy'] = move.get('accuracy')
                        row[f'{side}_move_details.priority'] = move.get('priority')
                    timeline_rows.append(row)

        return pd.DataFrame(p1_rows), pd.DataFrame(timeline_rows), pd.DataFrame(p2_rows)
    else:
        raise FileNotFoundError("Could not find pokemon_json_spacchettato.xlsx or train.jsonl!")

p1_team_df, timeline_df, p2_lead_df = load_competition_data()
type_chart_df = pd.read_csv(TYPE_CHART_PATH)

print(f"Loaded {p2_lead_df['battle_id'].nunique()} battles.")
print(f"P1 Team rows: {len(p1_team_df)} | Timeline rows: {len(timeline_df)} | P2 Lead rows: {len(p2_lead_df)}")

Loading raw JSONL data from /content/drive/MyDrive/1 SEMESTRE DATA SCIENCE/FUNDAMENTALS OF DATA SCIENCE/FDS PROJECT/train.jsonl...
Loaded 10000 battles.
P1 Team rows: 60000 | Timeline rows: 300000 | P2 Lead rows: 10000


In [ ]:
# ==============================================================================
# STEP 1C: BUILD TYPE EFFECTIVENESS ENGINE & METAGAME KNOWLEDGE BASE
# ==============================================================================
class CompetitiveMetaKnowledgeBase:
    """
    Learns strictly from training data (no external data leakage!):
    1. Exact Gen 2 Type Chart multipliers (0.0, 0.25, 0.5, 1.0, 2.0, 4.0)
    2. Internal Pokedex (maps Pokemon name -> types & base stats)
    3. Competitive Usage Priors (most used Pokemon & types in the meta)
    4. Team Type Co-occurrence Matrix (predicts remaining hidden types on P2's team)
    5. Competitive Move & Role Profiler (identifies Setup/Hazard/Status Pokemon & standard moves)
    """
    def __init__(self, type_chart_df):
        # 1. Build fast O(1) Type Multiplier Lookup
        self.type_matrix = defaultdict(lambda: defaultdict(lambda: 1.0))
        for _, row in type_chart_df.iterrows():
            atk = str(row['attacking_type']).lower().strip()
            dfd = str(row['defending_type']).lower().strip()
            self.type_matrix[atk][dfd] = float(row['multiplier'])

        self.pokedex = {}
        self.pokemon_usage_prob = {}
        self.type_usage_prob = {}
        self.type_cooccurrence = pd.DataFrame(0.0, index=ALL_TYPES, columns=ALL_TYPES)
        self.pokemon_moves_profile = defaultdict(Counter)
        self.pokemon_setup_ratio = {}

        # Define Competitive Setup, Hazard, Phazing & Recovery Moves in Gen 2 OU
        self.setup_and_support_moves = {
            'curse', 'swordsdance', 'bellydrum', 'amnesia', 'growth', 'agility', 'meditate',
            'spikes', 'toxic', 'thunderwave', 'stunspore', 'lovelykiss', 'sleeppowder',
            'hypnosis', 'spore', 'confuseray', 'attract', 'leechseed',
            'roar', 'whirlwind', 'haze', 'perishsong',
            'rest', 'sleeptalk', 'recover', 'milkdrink', 'softboiled', 'moonlight', 'synthesis',
            'reflect', 'lightscreen', 'safeguard', 'substitute', 'batonpass', 'encore', 'rapidspin'
        }

    def get_move_effectiveness(self, move_type, def_type_1, def_type_2=None):
        """Calculates exact Gen 2 damage multiplier (e.g., 0.0, 0.25, 0.5, 1.0, 2.0, 4.0)."""
        if pd.isna(move_type) or pd.isna(def_type_1):
            return 1.0
        m_type = str(move_type).lower().strip()
        t1 = str(def_type_1).lower().strip()
        mult = self.type_matrix[m_type][t1]

        if pd.notna(def_type_2) and str(def_type_2).lower().strip() not in ['nan', 'none', '']:
            t2 = str(def_type_2).lower().strip()
            mult *= self.type_matrix[m_type][t2]
        return mult

    def fit(self, p1_team_df, timeline_df, p2_lead_df):
        # A. Build Internal Pokedex from P1 full teams and P2 leads
        combined_pkmn = pd.concat([
            p1_team_df[['name', 'types', 'base_hp', 'base_atk', 'base_def', 'base_spa', 'base_spd', 'base_spe']],
            p2_lead_df[['name', 'types', 'base_hp', 'base_atk', 'base_def', 'base_spa', 'base_spd', 'base_spe']]
        ]).drop_duplicates(subset=['name'])

        for _, row in combined_pkmn.iterrows():
            raw_types = row['types']
            if isinstance(raw_types, str):
                import ast
                raw_types = ast.literal_eval(raw_types)

            # Filter out placeholder types like 'notype'
            clean_types = [t.lower().strip() for t in raw_types if t and str(t).lower().strip() not in ['notype', 'nan', 'none', '']]
            t1 = clean_types[0] if len(clean_types) > 0 else 'normal'
            t2 = clean_types[1] if len(clean_types) > 1 else None

            self.pokedex[row['name']] = {
                'type_1': t1,
                'type_2': t2,
                'types': clean_types,
                'base_hp': row['base_hp'],
                'base_atk': row['base_atk'],
                'base_def': row['base_def'],
                'base_spa': row['base_spa'],
                'base_spd': row['base_spd'],
                'base_spe': row['base_spe']
            }

        # B. Calculate Metagame Pokemon & Type Usage Frequencies (from P1 6-Pokemon teams)
        total_teams = p1_team_df['battle_id'].nunique()
        pkmn_counts = p1_team_df['name'].value_counts()
        self.pokemon_usage_prob = (pkmn_counts / total_teams).to_dict()

        type_counts = Counter()
        for _, grp in p1_team_df.groupby('battle_id'):
            team_types = set()
            for _, row in grp.iterrows():
                raw_types = row['types']
                if isinstance(raw_types, str):
                    import ast
                    raw_types = ast.literal_eval(raw_types)
                for t in raw_types:
                    t_clean = str(t).lower().strip()
                    if t_clean not in ['notype', 'nan', 'none', '']:
                        team_types.add(t_clean)
            for t in team_types:
                type_counts[t] += 1

            # C. Populate Team Type Co-occurrence Matrix
            for t_a in team_types:
                for t_b in team_types:
                    if t_a in ALL_TYPES and t_b in ALL_TYPES:
                        self.type_cooccurrence.loc[t_a, t_b] += 1.0

        self.type_usage_prob = {t: type_counts[t] / total_teams for t in ALL_TYPES}

        # Normalize Co-occurrence Matrix into Conditional Probabilities P(Type B in team | Type A in team)
        for t_a in ALL_TYPES:
            if type_counts[t_a] > 0:
                self.type_cooccurrence.loc[t_a, :] /= type_counts[t_a]

        # D. Profile Competitive Movesets & Setup vs. Offensive Roles from Timeline
        p1_moves = timeline_df[['p1_pokemon_state.name', 'p1_move_details.name', 'p1_move_details.type', 'p1_move_details.category']].dropna(subset=['p1_move_details.name'])
        p1_moves.columns = ['pokemon', 'move', 'move_type', 'category']

        p2_moves = timeline_df[['p2_pokemon_state.name', 'p2_move_details.name', 'p2_move_details.type', 'p2_move_details.category']].dropna(subset=['p2_move_details.name'])
        p2_moves.columns = ['pokemon', 'move', 'move_type', 'category']

        all_moves = pd.concat([p1_moves, p2_moves])

        setup_counts = Counter()
        total_move_counts = Counter()

        for _, row in all_moves.iterrows():
            pkmn = row['pokemon']
            mv = str(row['move']).lower().strip()
            self.pokemon_moves_profile[pkmn][mv] += 1
            total_move_counts[pkmn] += 1
            if mv in self.setup_and_support_moves or str(row['category']).lower() == 'status':
                setup_counts[pkmn] += 1

        for pkmn in total_move_counts:
            self.pokemon_setup_ratio[pkmn] = setup_counts[pkmn] / total_move_counts[pkmn]

        return self

    def predict_remaining_team_type_probs(self, revealed_pokemon_names):
        """
        Given the Pokemon revealed so far by a player, predicts the probability distribution
        of the remaining hidden types on their bench using competitive team co-occurrence.
        """
        revealed_types = set()
        for name in revealed_pokemon_names:
            if name in self.pokedex:
                revealed_types.update(self.pokedex[name]['types'])

        if not revealed_types:
            return self.type_usage_prob.copy()

        # Average conditional probability P(Target Type | Revealed Types)
        cond_probs = self.type_cooccurrence.loc[list(revealed_types), :].mean(axis=0).to_dict()
        return cond_probs

# Instantiate and fit our Knowledge Base on the dataset
meta_kb = CompetitiveMetaKnowledgeBase(type_chart_df).fit(p1_team_df, timeline_df, p2_lead_df)

# Sanity Check Output
print("=== TOP 10 MOST USED COMPETITIVE POKEMON (P(In Team)) ===")
for pkmn, prob in list(meta_kb.pokemon_usage_prob.items())[:10]:
    setup_pct = meta_kb.pokemon_setup_ratio.get(pkmn, 0.0) * 100
    top_moves = [m for m, _ in meta_kb.pokemon_moves_profile[pkmn].most_common(4)]
    print(f"- {pkmn.capitalize():<12} | Team Usage: {prob*100:5.1f}% | Setup/Support Role: {setup_pct:4.1f}% | Standard Moves: {top_moves}")

print("\n=== TOP 8 MOST USED TYPES ACROSS TEAMS ===")
for t, prob in sorted(meta_kb.type_usage_prob.items(), key=lambda x: x[1], reverse=True)[:8]:
    print(f"- {t.capitalize():<10}: present in {prob*100:5.1f}% of competitive teams")

=== TOP 10 MOST USED COMPETITIVE POKEMON (P(In Team)) ===
- Snorlax      | Team Usage:  82.2% | Setup/Support Role: 42.9% | Standard Moves: ['doubleedge', 'curse', 'bodyslam', 'rest']
- Cloyster     | Team Usage:  54.1% | Setup/Support Role: 62.2% | Standard Moves: ['spikes', 'surf', 'toxic', 'explosion']
- Zapdos       | Team Usage:  50.0% | Setup/Support Role: 28.0% | Standard Moves: ['thunder', 'hiddenpower', 'sleeptalk', 'thunderbolt']
- Raikou       | Team Usage:  31.8% | Setup/Support Role: 38.4% | Standard Moves: ['thunder', 'thunderbolt', 'hiddenpower', 'roar']
- Tyranitar    | Team Usage:  26.9% | Setup/Support Role: 19.5% | Standard Moves: ['rockslide', 'pursuit', 'roar', 'earthquake']
- Exeggutor    | Team Usage:  26.2% | Setup/Support Role: 38.7% | Standard Moves: ['psychic', 'sleeppowder', 'stunspore', 'gigadrain']
- Gengar       | Team Usage:  26.0% | Setup/Support Role: 21.1% | Standard Moves: ['icepunch', 'thunderbolt', 'dynamicpunch', 'hypnosis']
- Skarmory     | Team 

In [ ]:
# ==============================================================================
# STEP 2 (V2): UPGRADED TURN ENGINE WITH GEN-2 STAT/STAB DAMAGE & COUNTER MATRIX
# ==============================================================================
# Gen 2 Physical vs Special Type Split
PHYSICAL_TYPES = {'normal', 'fighting', 'flying', 'ground', 'rock', 'bug', 'ghost', 'poison', 'steel'}
SPECIAL_TYPES  = {'fire', 'water', 'grass', 'electric', 'psychic', 'ice', 'dragon', 'dark'}

def get_boost_multiplier(stage):
    """Converts Gen 2 stat stage (-6 to +6) into its exact combat multiplier."""
    stage = max(-6, min(6, int(stage)))
    return (2.0 + stage) / 2.0 if stage >= 0 else 2.0 / (2.0 - stage)

def extract_clean_types(types_val):
    if isinstance(types_val, (list, tuple, set)):
        return [str(t).lower().strip() for t in types_val if str(t).lower().strip() not in ['nan', 'none', 'notype', '']]
    elif pd.notna(types_val):
        t_str = str(types_val).lower().strip()
        if t_str not in ['nan', 'none', 'notype', '']:
            return [t_str]
    return []

def get_type_multiplier_from_list(meta_kb, move_type, def_types):
    if pd.isna(move_type) or not def_types:
        return 1.0
    m_type = str(move_type).lower().strip()
    if m_type in ['nan', 'none', 'threequestionmarks', '']:
        return 1.0
    mult = 1.0
    for dt in def_types:
        mult *= meta_kb.type_matrix[m_type][dt]
    return mult

def estimate_gen2_duel_threat(meta_kb, atk_name, atk_types, def_name, def_types, atk_boosts=None, def_boosts=None):
    """
    Computes best expected Gen 2 damage power between two Pokemon considering:
    - Attacker's STAB types + known competitive coverage move types
    - Gen 2 Physical/Special split (Base Atk vs Def, Base SpA vs SpD)
    - Active stat stage boosts (+1 Curse, +2 Swords Dance, etc.)
    - Exact Gen 2 Type Chart multiplier
    """
    if not def_types:
        return 1.0
    atk_info = meta_kb.pokedex.get(atk_name, {})
    def_info = meta_kb.pokedex.get(def_name, {})

    b_atk = atk_info.get('base_atk', 85) * get_boost_multiplier((atk_boosts or {}).get('atk', 0))
    b_spa = atk_info.get('base_spa', 85) * get_boost_multiplier((atk_boosts or {}).get('spa', 0))
    b_def = max(20.0, def_info.get('base_def', 85) * get_boost_multiplier((def_boosts or {}).get('def', 0)))
    b_spd = max(20.0, def_info.get('base_spd', 85) * get_boost_multiplier((def_boosts or {}).get('spd', 0)))

    candidate_types = set(atk_types)
    # Add common coverage move types for top meta Pokemon if known
    if not candidate_types:
        candidate_types = {'normal'}

    best_damage_index = 0.5
    for m_type in candidate_types:
        type_mult = get_type_multiplier_from_list(meta_kb, m_type, def_types)
        stab = 1.5 if m_type in atk_types else 1.0
        if m_type in PHYSICAL_TYPES:
            stat_ratio = b_atk / b_def
        else:
            stat_ratio = b_spa / b_spd
        best_damage_index = max(best_damage_index, type_mult * stab * stat_ratio)

    return best_damage_index

# Ensure Pokedex in meta_kb has clean types from the 'types' list column
for _, row in pd.concat([p1_team_df, p2_lead_df], ignore_index=True).drop_duplicates(subset=['name']).iterrows():
    p_types = extract_clean_types(row['types']) if 'types' in row else extract_clean_types([row.get('type_1'), row.get('type_2')])
    meta_kb.pokedex[row['name']] = {
        'types': p_types,
        'base_hp': float(row['base_hp']),
        'base_atk': float(row['base_atk']),
        'base_def': float(row['base_def']),
        'base_spa': float(row['base_spa']),
        'base_spd': float(row['base_spd']),
        'base_spe': float(row['base_spe']),
    }

print("Updated Pokedex with Gen 2 Physical/Special combat stats for", len(meta_kb.pokedex), "Pokemon.")

Updated Pokedex with Gen 2 Physical/Special combat stats for 239 Pokemon.


# step 3


In [ ]:
# ==============================================================================
# STEP 3 (V2): DEEP COMPETITIVE BATTLE FEATURE AGGREGATION
# ==============================================================================
def compute_turn31_expected_type_features_v2(turn_df, meta_kb, p1_sw_mod, p2_sw_mod, p1_nt_mod, p2_nt_mod):
    last_turns = turn_df.sort_values(['battle_id', 'turn']).groupby('battle_id', as_index=False).last()
    X_last = last_turns[TURN_FEATURE_COLS]

    p1_sw_prob = p1_sw_mod.predict_proba(X_last)[:, 1]
    p2_sw_prob = p2_sw_mod.predict_proba(X_last)[:, 1]

    p1_switch_type_probs = extract_multioutput_probs(p1_nt_mod, X_last)
    p2_switch_type_probs = extract_multioutput_probs(p2_nt_mod, X_last)

    p1_curr_types = last_turns[[f'p1_active_type_{t}' for t in ALL_TYPES]].values
    p2_curr_types = last_turns[[f'p2_active_type_{t}' for t in ALL_TYPES]].values

    p1_t31_type_probs = (1.0 - p1_sw_prob[:, None]) * p1_curr_types + p1_sw_prob[:, None] * p1_switch_type_probs
    p2_t31_type_probs = (1.0 - p2_sw_prob[:, None]) * p2_curr_types + p2_sw_prob[:, None] * p2_switch_type_probs

    M = np.ones((len(ALL_TYPES), len(ALL_TYPES)))
    for i, atk_t in enumerate(ALL_TYPES):
        for j, def_t in enumerate(ALL_TYPES):
            M[i, j] = meta_kb.type_matrix[atk_t][def_t]

    # Expected Turn-31 offensive multipliers
    p1_t31_expected_offense = np.einsum('bi,ij,bj->b', p1_t31_type_probs, M, p2_t31_type_probs)
    p2_t31_expected_offense = np.einsum('bi,ij,bj->b', p2_t31_type_probs, M, p1_t31_type_probs)

    # Also compute: If P1 stays in vs P2 switches (and vice versa)
    p1_vs_p2_switch_mult = np.einsum('bi,ij,bj->b', p1_curr_types, M, p2_switch_type_probs)
    p2_vs_p1_switch_mult = np.einsum('bi,ij,bj->b', p2_curr_types, M, p1_switch_type_probs)

    t31_df = pd.DataFrame({
        'battle_id': last_turns['battle_id'].values,
        't31_p1_switch_prob': p1_sw_prob,
        't31_p2_switch_prob': p2_sw_prob,
        't31_switch_prob_diff': p2_sw_prob - p1_sw_prob,
        't31_p1_expected_type_mult': p1_t31_expected_offense,
        't31_p2_expected_type_mult': p2_t31_expected_offense,
        't31_net_type_advantage': p1_t31_expected_offense - p2_t31_expected_offense,
        't31_p1_punish_switch_mult': p1_vs_p2_switch_mult,
        't31_p2_punish_switch_mult': p2_vs_p1_switch_mult,
        't30_active_net_duel_threat': last_turns['net_duel_threat'].values,
        't30_active_spe_advantage': last_turns['spe_advantage'].values,
        't30_p1_hp': last_turns['p1_hp'].values,
        't30_p2_hp': last_turns['p2_hp'].values,
        't30_active_hp_diff': last_turns['p1_hp'].values - last_turns['p2_hp'].values,
    })

    # Keep only the top high-impact competitive types on Turn 31 to prevent tree overfitting on rare types
    KEY_META_TYPES = ['normal', 'water', 'electric', 'steel', 'ground', 'rock', 'fighting', 'psychic', 'ghost', 'fire', 'ice']
    for t in KEY_META_TYPES:
        idx = ALL_TYPES.index(t)
        t31_df[f't31_p2_pred_type_{t}'] = p2_t31_type_probs[:, idx]
        t31_df[f't31_p1_pred_type_{t}'] = p1_t31_type_probs[:, idx]

    return t31_df, M


def build_battle_level_features_v2(p1_team_df, timeline_df, p2_lead_df, turn_df, t31_df, meta_kb, M_17x17):
    p1_team_info = defaultdict(dict)
    for _, row in p1_team_df.iterrows():
        b_id = row['battle_id']
        name = row['name']
        p_types = extract_clean_types(row['types']) if 'types' in row else extract_clean_types([row.get('type_1'), row.get('type_2')])
        p1_team_info[b_id][name] = {
            'types': p_types,
            'bst': float(row['base_hp'] + row['base_atk'] + row['base_def'] + row['base_spa'] + row['base_spd'] + row['base_spe']),
            'usage': meta_kb.pokemon_usage_prob.get(name, 0.0),
            'setup_role': meta_kb.pokemon_setup_ratio.get(name, 0.0),
            'base_spe': float(row['base_spe'])
        }

    # Global 30-turn aggregations
    turn_grp = turn_df.groupby('battle_id').agg(
        p1_mean_move_eff=('p1_move_eff_on_p2', 'mean'),
        p2_mean_move_eff=('p2_move_eff_on_p1', 'mean'),
        p1_se_turns=('p1_move_eff_on_p2', lambda s: (s >= 2.0).sum()),
        p2_se_turns=('p2_move_eff_on_p1', lambda s: (s >= 2.0).sum()),
        p1_resist_turns=('p1_resists_p2', 'sum'),
        p2_resist_turns=('p2_resists_p1', 'sum'),
        p1_tank_stay_turns=('p1_can_tank_and_stay', 'sum'),
        p2_tank_stay_turns=('p2_can_tank_and_stay', 'sum'),
        p1_mean_exp_threat=('p1_expected_threat_on_p2', 'mean'),
        p2_mean_exp_threat=('p2_expected_threat_on_p1', 'mean'),
        mean_net_duel_threat=('net_duel_threat', 'mean'),
        mean_spe_advantage=('spe_advantage', 'mean'),
        p1_faster_turns=('p1_is_faster', 'sum'),
        last_p1_bench_resist=('p1_bench_resist_count', 'last'),
    ).reset_index()

    # Late-game window (Turns 21 to 30) aggregations
    late_turns = turn_df[turn_df['turn'] >= 21].groupby('battle_id').agg(
        late_net_duel_threat=('net_duel_threat', 'mean'),
        late_p1_se_turns=('p1_move_eff_on_p2', lambda s: (s >= 2.0).sum()),
        late_p2_se_turns=('p2_move_eff_on_p1', lambda s: (s >= 2.0).sum()),
        late_spe_advantage=('spe_advantage', 'mean'),
    ).reset_index()

    # Status severity weights (when NOT a Rest/SleepTalk user)
    STATUS_SEVERITY = {'fnt': 1.0, 'frz': 0.85, 'slp': 0.65, 'tox': 0.45, 'brn': 0.40, 'par': 0.35, 'psn': 0.25, 'nostatus': 0.0}
    RECOVERY_MOVES = {'rest', 'recover', 'milkdrink', 'softboiled', 'moonlight', 'synthesis'}
    PHAZING_MOVES = {'roar', 'whirlwind'}

    battle_rows = []
    timeline_sorted = timeline_df.sort_values(['battle_id', 'turn']).reset_index(drop=True)

    for b_id, group in timeline_sorted.groupby('battle_id', sort=False):
        rows = group.to_dict('records')
        roster = p1_team_info[b_id]

        p1_hp = {name: 1.0 for name in roster}
        p1_status = {name: 'nostatus' for name in roster}
        p1_min_hp = {name: 1.0 for name in roster}
        p1_has_rest_talk = defaultdict(bool)

        p2_hp = {}
        p2_status = {}
        p2_min_hp = {}
        p2_has_rest_talk = defaultdict(bool)
        p1_revealed = set()

        p1_spikes_used, p2_spikes_used = 0, 0
        p1_setup_moves_used, p2_setup_moves_used = 0, 0
        p1_recovery_used, p2_recovery_used = 0, 0
        p1_phaze_used, p2_phaze_used = 0, 0
        p1_switches, p2_switches = 0, 0
        p1_late_switches, p2_late_switches = 0, 0
        p1_spikes_chip_taken, p2_spikes_chip_taken = 0, 0
        p1_missed_or_no_move, p2_missed_or_no_move = 0, 0

        prev_p1, prev_p2 = None, None
        hp_diff_turn_10, hp_diff_turn_20, hp_diff_turn_30 = 0.0, 0.0, 0.0

        for r in rows:
            t = int(r['turn'])
            p1_n = r['p1_pokemon_state.name']
            p2_n = r['p2_pokemon_state.name']
            p1_revealed.add(p1_n)

            c_p1_hp = float(r['p1_pokemon_state.hp_pct'])
            c_p2_hp = float(r['p2_pokemon_state.hp_pct'])

            # Detect switches & grounded Spikes chip damage
            if prev_p1 is not None and p1_n != prev_p1:
                p1_switches += 1
                if t >= 21: p1_late_switches += 1
                if p2_spikes_used > 0 and 'flying' not in roster.get(p1_n, {}).get('types', []):
                    p1_spikes_chip_taken += 1
            if prev_p2 is not None and p2_n != prev_p2:
                p2_switches += 1
                if t >= 21: p2_late_switches += 1
                if p1_spikes_used > 0 and 'flying' not in meta_kb.pokedex.get(p2_n, {}).get('types', []):
                    p2_spikes_chip_taken += 1
            prev_p1, prev_p2 = p1_n, p2_n

            p1_hp[p1_n] = c_p1_hp
            p2_hp[p2_n] = c_p2_hp
            p1_min_hp[p1_n] = min(p1_min_hp.get(p1_n, 1.0), c_p1_hp)
            p2_min_hp[p2_n] = min(p2_min_hp.get(p2_n, 1.0), c_p2_hp)

            p1_status[p1_n] = str(r['p1_pokemon_state.status']).lower().strip()
            p2_status[p2_n] = str(r['p2_pokemon_state.status']).lower().strip()

            m1 = str(r.get('p1_move_details.name') or '').lower().strip()
            m2 = str(r.get('p2_move_details.name') or '').lower().strip()

            if m1 in ['', 'nan', 'none']: p1_missed_or_no_move += 1
            if m2 in ['', 'nan', 'none']: p2_missed_or_no_move += 1

            if m1 == 'spikes': p1_spikes_used += 1
            if m2 == 'spikes': p2_spikes_used += 1
            if m1 in meta_kb.setup_and_support_moves: p1_setup_moves_used += 1
            if m2 in meta_kb.setup_and_support_moves: p2_setup_moves_used += 1
            if m1 in RECOVERY_MOVES: p1_recovery_used += 1
            if m2 in RECOVERY_MOVES: p2_recovery_used += 1
            if m1 in PHAZING_MOVES: p1_phaze_used += 1
            if m2 in PHAZING_MOVES: p2_phaze_used += 1
            if m1 in {'rest', 'sleeptalk'}: p1_has_rest_talk[p1_n] = True
            if m2 in {'rest', 'sleeptalk'}: p2_has_rest_talk[p2_n] = True

            if t in [10, 20, 30]:
                p1_tot_t = sum(p1_hp.values())
                p2_unrev_t = max(0, 6 - len(p2_hp))
                p2_tot_t = sum(p2_hp.values()) + p2_unrev_t * 1.0
                if t == 10: hp_diff_turn_10 = p1_tot_t - p2_tot_t
                elif t == 20: hp_diff_turn_20 = p1_tot_t - p2_tot_t
                elif t == 30: hp_diff_turn_30 = p1_tot_t - p2_tot_t

        # 1. Survival, Fainted & Weighted Status Severity
        p1_fainted = sum(1 for name, hp in p1_hp.items() if hp <= 0.0 or p1_status[name] == 'fnt')
        p2_fainted = sum(1 for name, hp in p2_hp.items() if hp <= 0.0 or p2_status[name] == 'fnt')
        p1_unrevealed = max(0, 6 - len(p1_revealed))
        p2_unrevealed = max(0, 6 - len(p2_hp))

        p1_total_hp = sum(p1_hp.values())
        p2_total_hp = sum(p2_hp.values()) + p2_unrevealed * 1.0

        # Calculate smart status severity penalty on ALIVE Pokemon
        p1_status_penalty, p2_status_penalty = 0.0, 0.0
        p1_statused_alive, p2_statused_alive = 0, 0
        for n, s in p1_status.items():
            if p1_hp[n] > 0 and s not in ['nostatus', 'fnt']:
                p1_statused_alive += 1
                if s == 'slp' and p1_has_rest_talk[n]:
                    p1_status_penalty += 0.15  # RestTalk sleep is much less punishing!
                else:
                    p1_status_penalty += STATUS_SEVERITY.get(s, 0.3)

        for n, s in p2_status.items():
            if p2_hp[n] > 0 and s not in ['nostatus', 'fnt']:
                p2_statused_alive += 1
                if s == 'slp' and p2_has_rest_talk[n]:
                    p2_status_penalty += 0.15
                else:
                    p2_status_penalty += STATUS_SEVERITY.get(s, 0.3)

        # Effective Health Pool (HP adjusted for status crippling!)
        p1_effective_hp = p1_total_hp - 0.5 * p1_status_penalty
        p2_effective_hp = p2_total_hp - 0.5 * p2_status_penalty

        # Snorlax & Top OU Tier Tracking
        p1_snorlax_hp = p1_hp.get('snorlax', 0.0)
        p2_snorlax_hp = p2_hp.get('snorlax', 0.822 * 1.0 if p2_unrevealed > 0 else 0.0)

        # 2. Best Bench Counter-Matchup Score (Min-Max Competitive Coverage!)
        # For each alive revealed P2 Pokemon, what is P1's BEST counter score among P1's alive Pokemon?
        p1_alive_mons = [(n, hp) for n, hp in p1_hp.items() if hp > 0]
        p2_alive_mons = [(n, hp) for n, hp in p2_hp.items() if hp > 0]

        p1_best_counter_scores = []
        for p2_n, p2_h in p2_alive_mons:
            p2_t = meta_kb.pokedex.get(p2_n, {}).get('types', [])
            best_score = -5.0
            for p1_n, p1_h in p1_alive_mons:
                p1_t = roster[p1_n]['types']
                off_threat = estimate_gen2_duel_threat(meta_kb, p1_n, p1_t, p2_n, p2_t)
                def_threat = estimate_gen2_duel_threat(meta_kb, p2_n, p2_t, p1_n, p1_t)
                best_score = max(best_score, (off_threat - def_threat) * p1_h)
            if best_score > -5.0:
                p1_best_counter_scores.append(best_score)

        p1_counter_coverage_mean = float(np.mean(p1_best_counter_scores)) if p1_best_counter_scores else 0.0
        p1_counter_coverage_min = float(np.min(p1_best_counter_scores)) if p1_best_counter_scores else 0.0

        # Whole-Team Remaining Type Matrix Score
        p1_rem_type_vec = np.zeros(len(ALL_TYPES))
        for name, hp in p1_hp.items():
            if hp > 0:
                for t in roster[name]['types']:
                    if t in ALL_TYPES:
                        p1_rem_type_vec[ALL_TYPES.index(t)] += hp

        p2_rem_type_vec = np.zeros(len(ALL_TYPES))
        for name, hp in p2_hp.items():
            if hp > 0:
                for t in meta_kb.pokedex.get(name, {}).get('types', []):
                    if t in ALL_TYPES:
                        p2_rem_type_vec[ALL_TYPES.index(t)] += hp

        p2_hidden_probs = meta_kb.predict_remaining_team_type_probs(set(p2_hp.keys()))
        for idx, t in enumerate(ALL_TYPES):
            p2_rem_type_vec[idx] += (p2_unrevealed / 6.0) * p2_hidden_probs.get(t, 0.0)

        p1_team_type_offense = float(p1_rem_type_vec @ M_17x17 @ p2_rem_type_vec)
        p2_team_type_offense = float(p2_rem_type_vec @ M_17x17 @ p1_rem_type_vec)

        # Surviving Base Stat Totals (BST) & Speed Tiers
        p1_alive_bst = sum(roster[n]['bst'] * hp for n, hp in p1_hp.items() if hp > 0)
        p2_alive_bst = sum(
            sum(meta_kb.pokedex.get(n, {}).get(s, 85) for s in ['base_hp', 'base_atk', 'base_def', 'base_spa', 'base_spd', 'base_spe']) * hp
            for n, hp in p2_hp.items() if hp > 0
        ) + p2_unrevealed * 535.0  # 535 is average OU BST

        p1_max_alive_spe = max([roster[n]['base_spe'] for n, hp in p1_hp.items() if hp > 0] or [0.0])
        p2_max_alive_spe = max([meta_kb.pokedex.get(n, {}).get('base_spe', 85) for n, hp in p2_hp.items() if hp > 0] or [100.0])

        p1_alive_meta_score = sum(roster[n]['usage'] * hp for n, hp in p1_hp.items() if hp > 0)
        p2_alive_meta_score = sum(meta_kb.pokemon_usage_prob.get(n, 0.1) * hp for n, hp in p2_hp.items() if hp > 0) + p2_unrevealed * 0.25

        # Final turn (Turn 30) boosts & volatile effects (Substitute, Leech Seed, Screens)
        last_r = rows[-1]
        p1_final_boosts = sum(int(last_r.get(f'p1_pokemon_state.boosts.{s}', 0)) for s in ['atk', 'def', 'spa', 'spd', 'spe'])
        p2_final_boosts = sum(int(last_r.get(f'p2_pokemon_state.boosts.{s}', 0)) for s in ['atk', 'def', 'spa', 'spd', 'spe'])
        p1_final_off_boosts = int(last_r.get('p1_pokemon_state.boosts.atk', 0)) + int(last_r.get('p1_pokemon_state.boosts.spa', 0))
        p2_final_off_boosts = int(last_r.get('p2_pokemon_state.boosts.atk', 0)) + int(last_r.get('p2_pokemon_state.boosts.spa', 0))

        p1_eff_str = str(last_r.get('p1_pokemon_state.effects', '')).lower()
        p2_eff_str = str(last_r.get('p2_pokemon_state.effects', '')).lower()
        p1_has_sub = int('substitute' in p1_eff_str)
        p2_has_sub = int('substitute' in p2_eff_str)

        battle_rows.append({
            'battle_id': b_id,
            'p1_total_hp': p1_total_hp,
            'p2_total_hp': p2_total_hp,
            'total_hp_diff': p1_total_hp - p2_total_hp,
            'total_hp_ratio': (p1_total_hp + 0.1) / (p2_total_hp + 0.1),
            'effective_hp_diff': p1_effective_hp - p2_effective_hp,
            'p1_fainted': p1_fainted,
            'p2_fainted': p2_fainted,
            'fainted_diff': p2_fainted - p1_fainted,
            'p1_unrevealed_count': p1_unrevealed,
            'p2_unrevealed_count': p2_unrevealed,
            'unrevealed_diff': p1_unrevealed - p2_unrevealed,
            'p1_statused_alive': p1_statused_alive,
            'p2_statused_alive': p2_statused_alive,
            'status_diff': p2_statused_alive - p1_statused_alive,
            'status_penalty_diff': p2_status_penalty - p1_status_penalty,
            'p1_snorlax_hp': p1_snorlax_hp,
            'p2_snorlax_hp': p2_snorlax_hp,
            'snorlax_hp_diff': p1_snorlax_hp - p2_snorlax_hp,
            'p1_spikes_active': int(p1_spikes_used > 0),
            'p2_spikes_active': int(p2_spikes_used > 0),
            'spikes_chip_diff': p2_spikes_chip_taken - p1_spikes_chip_taken,
            'p1_setup_moves_used': p1_setup_moves_used,
            'p2_setup_moves_used': p2_setup_moves_used,
            'setup_moves_diff': p1_setup_moves_used - p2_setup_moves_used,
            'recovery_moves_diff': p1_recovery_used - p2_recovery_used,
            'phaze_moves_diff': p1_phaze_used - p2_phaze_used,
            'p1_switches': p1_switches,
            'p2_switches': p2_switches,
            'switch_diff': p2_switches - p1_switches,
            'late_switch_diff': p2_late_switches - p1_late_switches,
            'action_efficiency_diff': p2_missed_or_no_move - p1_missed_or_no_move,
            'hp_diff_turn_10': hp_diff_turn_10,
            'hp_diff_turn_20': hp_diff_turn_20,
            'hp_momentum_20_to_30': hp_diff_turn_30 - hp_diff_turn_20,
            'hp_momentum_10_to_30': hp_diff_turn_30 - hp_diff_turn_10,
            'p1_team_type_offense': p1_team_type_offense,
            'p2_team_type_offense': p2_team_type_offense,
            'rem_team_type_advantage': p1_team_type_offense - p2_team_type_offense,
            'p1_counter_coverage_mean': p1_counter_coverage_mean,
            'p1_counter_coverage_min': p1_counter_coverage_min,
            'alive_bst_diff': p1_alive_bst - p2_alive_bst,
            'max_alive_spe_diff': p1_max_alive_spe - p2_max_alive_spe,
            'alive_meta_score_diff': p1_alive_meta_score - p2_alive_meta_score,
            'final_boost_diff': p1_final_boosts - p2_final_boosts,
            'final_off_boost_diff': p1_final_off_boosts - p2_final_off_boosts,
            'sub_active_diff': p1_has_sub - p2_has_sub,
        })

    battle_df = pd.DataFrame(battle_rows)
    battle_df = battle_df.merge(turn_grp, on='battle_id', how='left')
    battle_df = battle_df.merge(late_turns, on='battle_id', how='left').fillna(0.0)
    battle_df = battle_df.merge(t31_df, on='battle_id', how='left')

    if 'player_won' in p2_lead_df.columns:
        battle_df = battle_df.merge(p2_lead_df[['battle_id', 'player_won']], on='battle_id', how='left')

    return battle_df

t31_features_df, TYPE_MATRIX_17x17 = compute_turn31_expected_type_features_v2(
    turn_df, meta_kb, p1_switch_model, p2_switch_model, p1_next_type_model, p2_next_type_model
)
battle_features_df = build_battle_level_features_v2(
    p1_team_df, timeline_df, p2_lead_df, turn_df, t31_features_df, meta_kb, TYPE_MATRIX_17x17
)
print(f"Upgraded V2 Battle-Level Dataset Shape: {battle_features_df.shape}")

Upgraded V2 Battle-Level Dataset Shape: (10000, 101)


```markdown
# ==============================================================================
# STEP 4: 5-FOLD STRATIFIED CROSS-VALIDATION, ENSEMBLING & SUBMISSION
# ==============================================================================
```

In [ ]:
# ==============================================================================
# STEP 4 (V2): TEST SET PROCESSING, 10-FOLD STACKED ENSEMBLE & PRIOR CALIBRATION
# ==============================================================================
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

if os.path.exists(TEST_JSONL_PATH):
    print("Building V2 features for test set...")
    if 'test_p1_df' not in globals():
        test_p1_df, test_timeline_df, test_p2_df = load_test_data(TEST_JSONL_PATH)
    test_turn_df = build_turn_level_dataset_v2(test_p1_df, test_timeline_df, meta_kb)
    test_t31_df, _ = compute_turn31_expected_type_features_v2(
        test_turn_df, meta_kb, p1_switch_model, p2_switch_model, p1_next_type_model, p2_next_type_model
    )
    test_battle_features_df = build_battle_level_features_v2(
        test_p1_df, test_timeline_df, test_p2_df, test_turn_df, test_t31_df, meta_kb, TYPE_MATRIX_17x17
    )
    print(f"Upgraded V2 Test Dataset Shape: {test_battle_features_df.shape}")

FEATURE_COLS = [c for c in battle_features_df.columns if c not in ['battle_id', 'player_won']]
X = battle_features_df[FEATURE_COLS].copy()
y = battle_features_df['player_won'].astype(int).values
X_test = test_battle_features_df[FEATURE_COLS].copy() if test_battle_features_df is not None else None

# Use 10-Fold Stratified CV for stronger regularization and higher training data per fold (90% vs 80%)
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

test_preds_lgb = np.zeros(len(X_test)) if X_test is not None else None
test_preds_xgb = np.zeros(len(X_test)) if X_test is not None else None
test_preds_cat = np.zeros(len(X_test)) if X_test is not None else None
feature_importances = np.zeros(len(FEATURE_COLS))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]

    # 1. Tuned LightGBM (shallower leaves + stronger regularization to prevent overfitting)
    model_lgb = LGBMClassifier(
        n_estimators=1200,
        learning_rate=0.02,
        num_leaves=23,
        min_child_samples=40,
        subsample_for_bin=50000,
        colsample_bytree=0.7,
        subsample=0.8,
        reg_alpha=2.0,
        reg_lambda=2.0,
        random_state=42 + fold,
        verbose=-1
    )
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[__import__('lightgbm').early_stopping(stopping_rounds=60, verbose=False)]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va)[:, 1]
    feature_importances += model_lgb.feature_importances_ / N_SPLITS

    # 2. Tuned XGBoost
    model_xgb = XGBClassifier(
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=10,
        colsample_bytree=0.7,
        subsample=0.8,
        reg_alpha=2.0,
        reg_lambda=2.0,
        eval_metric='logloss',
        early_stopping_rounds=60,
        random_state=42 + fold,
        n_jobs=-1
    )
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_va)[:, 1]

    # 3. Tuned CatBoost
    model_cat = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=5.0,
        random_seed=42 + fold,
        verbose=False
    )
    model_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=60, verbose=False)
    oof_cat[val_idx] = model_cat.predict_proba(X_va)[:, 1]

    if X_test is not None:
        test_preds_lgb += model_lgb.predict_proba(X_test)[:, 1] / N_SPLITS
        test_preds_xgb += model_xgb.predict_proba(X_test)[:, 1] / N_SPLITS
        test_preds_cat += model_cat.predict_proba(X_test)[:, 1] / N_SPLITS

# Train a Logistic Regression Meta-Learner (Stacking) out-of-fold to find optimal model weights
meta_X_oof = np.column_stack([oof_lgb, oof_xgb, oof_cat])
oof_stacked = np.zeros(len(X))
meta_weights = np.zeros(3)

for train_idx, val_idx in skf.split(meta_X_oof, y):
    meta_clf = LogisticRegression(C=1.0)
    meta_clf.fit(meta_X_oof[train_idx], y[train_idx])
    oof_stacked[val_idx] = meta_clf.predict_proba(meta_X_oof[val_idx])[:, 1]
    meta_weights += meta_clf.coef_[0] / N_SPLITS

print("\n" + "="*65)
print("V2 10-FOLD OUT-OF-FOLD (OOF) VALIDATION RESULTS (10,000 BATTLES)")
print("="*65)
for name, preds in [
    ('LightGBM (V2)', oof_lgb),
    ('XGBoost (V2)', oof_xgb),
    ('CatBoost (V2)', oof_cat),
    ('Stacked Meta-Ensemble', oof_stacked)
]:
    acc = accuracy_score(y, (preds >= 0.50).astype(int))
    auc = roc_auc_score(y, preds)
    print(f"{name:<22} -> OOF Accuracy (@0.50): {acc*100:.2f}% | OOF ROC-AUC: {auc:.4f}")

# Generate Properly Calibrated Test Predictions (Matching True Training Win-Rate Prior!)
if test_battle_features_df is not None:
    meta_clf_full = LogisticRegression(C=1.0).fit(meta_X_oof, y)
    meta_X_test = np.column_stack([test_preds_lgb, test_preds_xgb, test_preds_cat])
    test_stacked_probs = meta_clf_full.predict_proba(meta_X_test)[:, 1]

    # Calibrate cutoff so Test Set P1 Win Rate matches the true Training Set Win Rate (~50%)
    train_win_rate = y.mean()
    prior_calibrated_thresh = float(np.quantile(test_stacked_probs, 1.0 - train_win_rate))

    # Blend standard 0.500 cutoff with prior-calibrated cutoff for maximum stability
    safe_thresh = 0.5 * 0.500 + 0.5 * prior_calibrated_thresh
    final_test_predictions = (test_stacked_probs >= safe_thresh).astype(int)

    submission_df = pd.DataFrame({
        'battle_id': test_battle_features_df['battle_id'].astype(int),
        'player_won': final_test_predictions
    }).sort_values('battle_id').reset_index(drop=True)

    submission_path = os.path.join(DATA_DIR, "submission_v2_26_10.csv")
    submission_df.to_csv(submission_path, index=False)
    print(f"\nTraining True Win Rate: {train_win_rate*100:.2f}% | Safe Threshold: {safe_thresh:.4f}")
    print(f"New Predicted Win Rate for Player 1 in Test Set: {submission_df['player_won'].mean()*100:.2f}% (Fixed from 59.76%!)")
    print(f"Saved upgraded '{submission_path}' successfully!")

Building V2 features for test set...
Upgraded V2 Test Dataset Shape: (5000, 100)

V2 10-FOLD OUT-OF-FOLD (OOF) VALIDATION RESULTS (10,000 BATTLES)
LightGBM (V2)          -> OOF Accuracy (@0.50): 67.74% | OOF ROC-AUC: 0.7472
XGBoost (V2)           -> OOF Accuracy (@0.50): 67.87% | OOF ROC-AUC: 0.7513
CatBoost (V2)          -> OOF Accuracy (@0.50): 67.91% | OOF ROC-AUC: 0.7500
Stacked Meta-Ensemble  -> OOF Accuracy (@0.50): 68.14% | OOF ROC-AUC: 0.7514

Training True Win Rate: 50.00% | Safe Threshold: 0.5055
New Predicted Win Rate for Player 1 in Test Set: 50.80% (Fixed from 59.76%!)
Saved upgraded '/content/drive/MyDrive/1 SEMESTRE DATA SCIENCE/FUNDAMENTALS OF DATA SCIENCE/FDS PROJECT/submission_v2_26_10.csv' successfully!
